# 43 — Qualification Detection
**Goal:** Extract degree requirements, year requirements, and certifications.

Qualifications are the *hard gates* of a JD: education level and years of experience. They are also among the most formulaic text in any posting — "MS/PhD in Computer Science", "5+ years experience" — which makes them ideal targets for **regex extraction**. This chapter pulls out (a) which degree levels are required and (b) how many years of experience, using two small, readable pattern sets.

**Why it matters for resumes / ATS:** degree and years are the closest thing a JD has to boolean filters. If the JD requires an MS and 5 years, a candidate with a BS and 2 years can be screened out *before* expensive semantic matching. Extracting these numbers as structured fields is also what lets an ATS answer recruiter queries like "show me senior ML roles that require a PhD" — and it is the natural complement to the skill and responsibility signals from Ch. 41–42.

## 1. Degree Requirement Extraction

Degrees appear in many spellings — "PhD", "Ph.D.", "doctorate" — so the extractor maps every alias to one of three canonical levels: `phd`, `masters`, `bachelors`. Matching uses **word-boundary regexes** (`\b` around each alias) so that "MS" in "MS/PhD" matches while "MS" inside "MSc" or "MSc." does not.

**What the code does:**
- `degrees` is a dict of level → alias list ("masters": ["masters", "ms", "m.s.", "m.tech", "m.sc"], …).
- For each alias it builds `\b` + `re.escape(alias)` + `\b` and searches with `re.IGNORECASE`.
- A level is recorded once (the inner `break` stops after the first matching alias) and the result list preserves dict order.

**Expected:** on the sample JD — "MS/PhD in Computer Science or related field" — this returns `['phd', 'masters']`. Note the order follows the *dict* (`phd` first), not the order of appearance in the text. Both aliases match the same string because the `\b` boundaries anchor each alias independently; "MS" and "PhD" are separated by a slash, so both fire. If the JD said "Master's degree" the alias list would need "master's" added — a reminder that the vocabulary is the real product here.

In [ ]:
import re

def extract_degree_requirements(jd_text):
    degrees = {
        "phd": ["phd", "doctorate", "ph.d"],
        "masters": ["masters", "ms", "m.s.", "m.tech", "m.sc"],
        "bachelors": ["bachelors", "bs", "b.s.", "b.tech", "b.e.", "b.sc"],
    }
    found = []
    for level, keywords in degrees.items():
        for kw in keywords:
            if re.search(r"\\b" + re.escape(kw) + r"\\b", jd_text, re.IGNORECASE):
                found.append(level)
                break
    return found

print(f"Degree requirements: {extract_degree_requirements(jd)}")

## 2. Experience Year Extraction

Years-of-experience requirements come in two word orders: "5+ years experience" and "experience: 5+ years". `extract_years_required()` tries **two regex patterns** in sequence and returns the first match's number, or `None` if neither fits.

**What the code does:**
- Pattern 1 anchors the number first: `(\d+)[+]?\s*years?\s+(?:of\s+)?(?:experience|exp)` — matches "5+ years experience", "3 years of experience", "2 yrs exp" variants.
- Pattern 2 anchors "experience" first and looks ahead for a number within ~20 chars: `(?:experience|exp)[^\n]{0,20}(\d+)[+]?\s*years?` — matches "experience: 5+ years".
- Returns `int(match.group(1))`; the `[+]?` handles the "+" in "5+" while `\s*` absorbs spacing.

**Expected:** on "5+ years experience in data science" the first pattern fires and the function returns `5`. A JD with no year phrasing returns `None` — a real distinction for the matcher (unknown ≠ zero years). The two-pattern design is the standard way to stay robust to word order without writing one giant regex; ranges like "5–7 years" would need a third pattern, and that is the ongoing maintenance cost of regex extraction.

In [ ]:
def extract_years_required(jd_text):
    """Extract years of experience required."""
    patterns = [
        r"(\d+)[+]?\s*(?:\+)?\s*years?\s+(?:of\s+)?(?:experience|exp)",
        r"(?:experience|exp)[^\n]{0,20}(\d+)[+]?\s*years?",
    ]
    for pat in patterns:
        match = re.search(pat, jd_text, re.IGNORECASE)
        if match:
            return int(match.group(1))
    return None

print(f"Years required: {extract_years_required(jd)}")

## Summary: Regex extracts degree and experience requirements. Combine for qualification matching.

**Degree and years are the JD's boolean filters — cheap to extract, decisive in screening — and regex is the right tool because the phrasing is formulaic.**

The two extractors show the whole regex mindset: alias dictionaries plus `\b`-anchored, case-insensitive patterns for degrees; ordered patterns plus `[+]?`-tolerant number capture for years. The verified outputs on the sample — `['phd', 'masters']` and `5` — are exactly the structured fields a matcher needs. Combining them with skills (Ch. 41) and responsibilities (Ch. 42) yields a complete JD profile: what you must know, what you must have done, and what you must hold. Ch. 44 shifts from *extraction* to *ranking* — deciding which of a JD's many keywords actually matter, via TF-IDF and embeddings.